# Pipelines

In [32]:
# importar as bibliotecas relevantes

import io
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sys

from pandas import Timestamp
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

In [33]:
# ler os dados

dados = pd.read_csv("./Aulas/titanic.csv")
dados

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [34]:
# Selecção dos dados

dados.set_index("PassengerId", inplace=True, drop=True)
dados["FamilySize"] = dados.SibSp + dados.Parch
dados.drop(["SibSp", "Parch", "Ticket", "Name", "Cabin"], inplace=True, axis=1)
dados.head()

,Survived,Pclass,Sex,Age,Fare,Embarked,FamilySize
PassengerId,,,,,,,
1,0,3,male,22.0,7.2500,S,1
2,1,1,female,38.0,71.2833,C,1
3,1,3,female,26.0,7.9250,S,0
4,1,1,female,35.0,53.1000,S,1
5,0,3,male,35.0,8.0500,S,0


In [35]:
dados.info()

<class 'pandas.core.frame.DataFrame'>
Index: 891 entries, 1 to 891
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Survived    891 non-null    int64  
 1   Pclass      891 non-null    int64  
 2   Sex         891 non-null    object 
 3   Age         714 non-null    float64
 4   Fare        891 non-null    float64
 5   Embarked    889 non-null    object 
 6   FamilySize  891 non-null    int64  
dtypes: float64(2), int64(3), object(2)
memory usage: 55.7+ KB


In [36]:
# Nomes das colunas numéricas e categóricas
num = dados.select_dtypes(include="number").columns 
cat = dados.select_dtypes(include="object").columns
print("num:", num)
print("cat:", cat)

num: Index(['Survived', 'Pclass', 'Age', 'Fare', 'FamilySize'], dtype='object')
cat: Index(['Sex', 'Embarked'], dtype='object')


# Pipeline para os dados Numéricos

In [37]:
num_transformer = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', RobustScaler())
])

## Pipeline para os dados Categóricos

In [ ]:
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(sparse_output=False, drop='first'))
])

['__abstractmethods__',
 '__annotations__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__len__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__sklearn_clone__',
 '__sklearn_is_fitted__',
 '__sklearn_tags__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 '_build_request_for_signature',
 '_can_fit_transform',
 '_can_inverse_transform',
 '_can_transform',
 '_check_method_params',
 '_doc_link_module',
 '_doc_link_template',
 '_doc_link_url_param_generator',
 '_estimator_type',
 '_final_estimator',
 '_fit',
 '_get_default_requests',
 '_get_doc_link',
 '_get_metadata_for_step',
 '_get_metadata_request',
 '_get_param_names',
 '_get_params',
 '_get_params_html',
 '_html_repr',
 '_iter',
 '_log

## Aplicar as  transformações

In [44]:
pre_processamento = ColumnTransformer(transformers=[
    ('num', num_transformer, num),
    ('cat', cat_transformer, cat)
])

In [45]:
dados_final = pre_processamento.fit_transform(dados)
dados_final

array([[ 0.        ,  0.        , -0.59223982, ...,  1.        ,
         0.        ,  1.        ],
       [ 1.        , -2.        ,  0.63852941, ...,  0.        ,
         0.        ,  0.        ],
       [ 1.        ,  0.        , -0.28454751, ...,  0.        ,
         0.        ,  1.        ],
       ...,
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  1.        ],
       [ 1.        , -2.        , -0.28454751, ...,  1.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.17699095, ...,  1.        ,
         1.        ,  0.        ]], shape=(891, 8))

In [ ]:
cat_columns = pre_processamento.named_transformers_["cat"].named_steps['ohe'].get_feature_names_out(cat)
cat_columns

array(['Sex_male', 'Embarked_Q', 'Embarked_S'], dtype=object)

In [58]:
colunas = list(num) + list(cat_columns)
colunas

['Survived',
 'Pclass',
 'Age',
 'Fare',
 'FamilySize',
 'Sex_male',
 'Embarked_Q',
 'Embarked_S']

In [59]:
dados_final = pd.DataFrame(dados_final, columns=colunas)
dados_final.describe()

,Survived,Pclass,Age,Fare,FamilySize,Sex_male,Embarked_Q,Embarked_S
count,891.000000,891.000000,8.910000e+02,891.000000,891.000000,891.000000,891.000000,891.000000
mean,0.383838,-0.691358,2.153160e-16,0.768745,0.904602,0.647587,0.086420,0.725028
std,0.486592,0.836071,1.000155e+00,2.152200,1.613459,0.477990,0.281141,0.446751
min,0.000000,-2.000000,-2.252240e+00,-0.626005,0.000000,0.000000,0.000000,0.000000
25%,0.000000,-1.000000,-5.922398e-01,-0.283409,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000e+00,0.000000,0.000000,1.000000,0.000000,1.000000
75%,1.000000,0.000000,4.077602e-01,0.716591,1.000000,1.000000,0.000000,1.000000
max,1.000000,0.000000,3.869299e+00,21.562738,10.000000,1.000000,1.000000,1.000000


In [61]:
dados_final.head()

,Survived,Pclass,Age,Fare,FamilySize,Sex_male,Embarked_Q,Embarked_S
0,0.0,0.0,-0.592240,-0.312011,1.0,1.0,0.0,1.0
1,1.0,-2.0,0.638529,2.461242,1.0,0.0,0.0,0.0
2,1.0,0.0,-0.284548,-0.282777,0.0,0.0,0.0,1.0
3,1.0,-2.0,0.407760,1.673732,1.0,0.0,0.0,1.0
4,0.0,0.0,0.407760,-0.277363,0.0,1.0,0.0,1.0


In [64]:
# Gravar os dados
dados_final.to_csv("titanic_preprocessado_scaler.csv", sep=",", index=False)